In [ ]:
import os
print("✅ Working directory:", os.getcwd())

In [ ]:
import subprocess

subprocess.run(["pip", "install", "transformers"])
subprocess.run(["pip", "install", "torch"])
subprocess.run(["pip", "install", "bertopic"])
subprocess.run(["pip", "install", "sentence-transformers"])

print("✅ All libraries installed")

In [ ]:
from transformers import pipeline

print("Loading FinBERT model... (may take 1-2 minutes first time)")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True
)

print("✅ FinBERT loaded successfully")

# Quick test
test_sentence = "Our revenue growth has been strong and margins are expanding."
result = finbert(test_sentence)
print(f"\nTest sentence: '{test_sentence}'")
print(f"Result: {result}")

In [ ]:
import pandas as pd

df = pd.read_csv("data/structured_utterances.csv")
print(f"Loaded {len(df)} utterances")

def get_sentiment(text):
    try:
        # FinBERT has 512 token limit — truncate long texts
        text = str(text)[:512]
        scores = finbert(text)[0]
        
        # scores is a list of dicts with label and score
        score_dict = {s['label']: s['score'] for s in scores}
        
        # Get the dominant sentiment
        dominant = max(score_dict, key=score_dict.get)
        confidence = score_dict[dominant]
        
        return pd.Series({
            'sentiment': dominant,
            'positive_score': score_dict.get('positive', 0),
            'negative_score': score_dict.get('negative', 0),
            'neutral_score': score_dict.get('neutral', 0),
            'confidence': confidence
        })
    except Exception as e:
        return pd.Series({
            'sentiment': 'neutral',
            'positive_score': 0,
            'negative_score': 0,
            'neutral_score': 1,
            'confidence': 0
        })

print("Running FinBERT on all utterances...")
print("This will take 3-5 minutes...")

# Run on Management utterances first (most important for analysis)
mgmt_df = df[df['role'] == 'Management'].copy()
sentiment_cols = mgmt_df['utterance'].apply(get_sentiment)
mgmt_df = pd.concat([mgmt_df.reset_index(drop=True), sentiment_cols], axis=1)

print(f"\n✅ Sentiment analysis complete for {len(mgmt_df)} management utterances")
print(f"\nSentiment breakdown:")
print(mgmt_df['sentiment'].value_counts())

In [ ]:
# Check raw FinBERT output on a sample utterance
sample = mgmt_df['utterance'].iloc[0]
print("Sample text:")
print(sample[:300])
print("\nRaw FinBERT output:")
raw_output = finbert(str(sample)[:512])
print(raw_output)

In [ ]:
from transformers import pipeline

# Newer transformers uses top_k=None instead of return_all_scores=True
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None        # ← This replaces return_all_scores=True
)

# Verify it now returns all 3 scores
test = "Our revenue growth has been strong and margins are expanding."
output = finbert(test)
print("Test output:")
print(output)

In [ ]:
def get_sentiment_v2(text):
    try:
        text = str(text)[:512]
        scores = finbert(text)[0]
        score_dict = {s['label'].lower(): s['score'] for s in scores}
        dominant = max(score_dict, key=score_dict.get)
        
        return pd.Series({
            'sentiment': dominant,
            'positive_score': score_dict.get('positive', 0),
            'negative_score': score_dict.get('negative', 0),
            'neutral_score': score_dict.get('neutral', 0),
            'confidence': score_dict[dominant]
        })
    except Exception as e:
        print(f"Error: {e}")
        return pd.Series({
            'sentiment': 'neutral',
            'positive_score': 0,
            'negative_score': 0,
            'neutral_score': 1,
            'confidence': 0
        })

print("Re-running FinBERT with fix...")
sentiment_cols = mgmt_df['utterance'].apply(get_sentiment_v2)
mgmt_df = pd.concat([
    mgmt_df[['ticker','quarter','speaker','role','utterance','word_count']].reset_index(drop=True),
    sentiment_cols
], axis=1)

print("\n✅ Done!")
print("\nSentiment breakdown:")
print(mgmt_df['sentiment'].value_counts())
print("\nAverage scores by company:")
print(mgmt_df.groupby('ticker')[['positive_score','negative_score','neutral_score']].mean().round(3))

In [ ]:
# Save what we have
mgmt_df.to_csv("data/management_sentiment.csv", index=False)
print("✅ Saved management sentiment")

# Check why HUL is missing
print("\nAll tickers in mgmt_df:")
print(mgmt_df['ticker'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

quarter_order = ['Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25']
companies = mgmt_df['ticker'].unique()

fig, axes = plt.subplots(1, len(companies), figsize=(6*len(companies), 5))

if len(companies) == 1:
    axes = [axes]

for i, company in enumerate(companies):
    data = mgmt_df[mgmt_df['ticker'] == company].groupby('quarter')[
        ['positive_score', 'negative_score', 'neutral_score']
    ].mean()
    
    data = data.reindex([q for q in quarter_order if q in data.index])
    
    axes[i].plot(data.index, data['positive_score'], 
                 marker='o', color='green', label='Positive', linewidth=2)
    axes[i].plot(data.index, data['negative_score'], 
                 marker='o', color='red', label='Negative', linewidth=2)
    axes[i].plot(data.index, data['neutral_score'], 
                 marker='o', color='gray', label='Neutral', linewidth=2, linestyle='--')
    axes[i].set_title(f'{company} — Management Sentiment FY25', fontsize=12)
    axes[i].set_xlabel('Quarter')
    axes[i].set_ylabel('Average Score')
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("data/sentiment_trend.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

In [ ]:
import torch

# Check if MPS (Apple Silicon GPU) is available
if torch.backends.mps.is_available():
    device = "mps"
    print("✅ Apple M4 Neural Engine detected — using MPS acceleration")
else:
    device = "cpu"
    print("Using CPU")

# Reload FinBERT with M5 acceleration
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=device    # ← This uses your M4 chip
)

In [ ]:

# Check what's happening with HUL
print(mgmt_df['ticker'].value_counts())
print("\nRows where ticker is nan:")
print(mgmt_df[mgmt_df['ticker'].isna()][['speaker', 'utterance']].head(3))

In [ ]:
# Step 1 — Drop empty rows
mgmt_df = mgmt_df.dropna(subset=['ticker', 'utterance'])
print(f"After dropping NaN rows: {len(mgmt_df)} rows")
print(mgmt_df['ticker'].value_counts())

In [ ]:
# Step 2 — Check what HUL speakers look like in original data
df_full = pd.read_csv("data/structured_utterances.csv")
hul_df = df_full[df_full['ticker'] == 'HUL']
print(f"\nHUL total utterances: {len(hul_df)}")
print(f"\nHUL role breakdown:")
print(hul_df['role'].value_counts())
print(f"\nHUL unique speakers:")
print(hul_df['speaker'].unique())

In [ ]:
import pandas as pd

df = pd.read_csv("data/structured_utterances.csv")

def classify_role_v3(speaker_name):
    speaker_lower = str(speaker_name).lower()
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'sub', 'speakers']
    
    management_keywords = [
        'ceo', 'cfo', 'coo', 'cto', 'chairman', 'director',
        'president', 'managing', 'executive', 'officer',
        'jagdishan', 'vaidyanathan', 'lakhpatwala',
        'chandrasekaran', 'padmanabhan', 'mistry',
        'jawa', 'tiwari', 'mulgaonkar', 'nair', 'srinivas',
        'mehta', 'khattar', 'gupta', 'rohit', 'ritesh', 'yogesh'
    ]
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in management_keywords):
        return 'Management'
    else:
        return 'Analyst'

df['role'] = df['speaker'].apply(classify_role_v3)
df.to_csv("data/structured_utterances.csv", index=False)

print("Updated role breakdown:")
print(df.groupby('role').size())
print("\nHUL management count:")
print(df[(df['ticker']=='HUL') & (df['role']=='Management')].shape[0])

In [ ]:
mgmt_df = df[df['role'] == 'Management'].copy()
print(f"Management utterances by company:")
print(mgmt_df['ticker'].value_counts())

print("\nRunning FinBERT...")
sentiment_cols = mgmt_df['utterance'].apply(get_sentiment_v2)
mgmt_df = pd.concat([
    mgmt_df[['ticker','quarter','speaker','role','utterance','word_count']].reset_index(drop=True),
    sentiment_cols
], axis=1)

mgmt_df = mgmt_df.dropna(subset=['ticker','utterance'])
mgmt_df.to_csv("data/management_sentiment.csv", index=False)

print("\n✅ Done!")
print("\nSentiment by company:")
print(mgmt_df.groupby(['ticker','sentiment']).size())

In [ ]:
import matplotlib.pyplot as plt

quarter_order = ['Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25']
companies = ['HDFC', 'HUL', 'Tata']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, company in enumerate(companies):
    data = mgmt_df[mgmt_df['ticker'] == company].groupby('quarter')[
        ['positive_score', 'negative_score', 'neutral_score']
    ].mean().reindex([q for q in quarter_order if q in 
                      mgmt_df[mgmt_df['ticker']==company]['quarter'].values])
    
    axes[i].plot(data.index, data['positive_score'], 
                 marker='o', color='green', label='Positive', linewidth=2)
    axes[i].plot(data.index, data['negative_score'], 
                 marker='o', color='red', label='Negative', linewidth=2)
    axes[i].plot(data.index, data['neutral_score'], 
                 marker='o', color='gray', label='Neutral', 
                 linewidth=2, linestyle='--')
    axes[i].set_title(f'{company} — Management Sentiment FY25', fontsize=12)
    axes[i].set_xlabel('Quarter')
    axes[i].set_ylabel('Average Score')
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("data/sentiment_trend_final.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Final chart saved")

In [ ]:
import torch
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using: {device}")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=device
)
print("✅ FinBERT loaded")

In [ ]:
import pandas as pd

mgmt_df = pd.read_csv("data/management_sentiment.csv")
print(f"Total rows: {len(mgmt_df)}")

def get_sentiment_v2(text):
    try:
        text = str(text)[:512]
        scores = finbert(text)[0]
        score_dict = {s['label'].lower(): s['score'] for s in scores}
        dominant = max(score_dict, key=score_dict.get)
        return pd.Series({
            'sentiment': dominant,
            'positive_score': score_dict.get('positive', 0),
            'negative_score': score_dict.get('negative', 0),
            'neutral_score': score_dict.get('neutral', 0),
            'confidence': score_dict[dominant]
        })
    except:
        return pd.Series({
            'sentiment': 'neutral',
            'positive_score': 0,
            'negative_score': 0,
            'neutral_score': 1,
            'confidence': 0
        })

print("Running FinBERT on all 325 rows...")
sentiment_cols = mgmt_df['utterance'].apply(get_sentiment_v2)

# Overwrite sentiment columns with fresh results
mgmt_df['sentiment'] = sentiment_cols['sentiment']
mgmt_df['positive_score'] = sentiment_cols['positive_score']
mgmt_df['negative_score'] = sentiment_cols['negative_score']
mgmt_df['neutral_score'] = sentiment_cols['neutral_score']
mgmt_df['confidence'] = sentiment_cols['confidence']

mgmt_df.to_csv("data/management_sentiment.csv", index=False)
print(f"\n✅ Done!")
print(f"Sentiment breakdown:")
print(mgmt_df['sentiment'].value_counts())
print(f"\nRows with sentiment: {mgmt_df['sentiment'].notna().sum()}")

In [ ]:
import os
import torch
import pandas as pd
from transformers import pipeline


device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using: {device}")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=device
)
print("✅ FinBERT loaded")

In [ ]:
df = pd.read_csv("data/structured_utterances.csv")
mgmt_df = df[df['role'] == 'Management'].copy().reset_index(drop=True)
print(f"Management utterances: {len(mgmt_df)}")
print(mgmt_df['ticker'].value_counts())

def get_sentiment(text):
    try:
        text = str(text)[:512]
        scores = finbert(text)[0]
        score_dict = {s['label'].lower(): s['score'] for s in scores}
        dominant = max(score_dict, key=score_dict.get)
        return pd.Series({
            'sentiment': dominant,
            'positive_score': score_dict.get('positive', 0),
            'negative_score': score_dict.get('negative', 0),
            'neutral_score': score_dict.get('neutral', 0),
            'confidence': score_dict[dominant]
        })
    except:
        return pd.Series({
            'sentiment': 'neutral',
            'positive_score': 0,
            'negative_score': 0,
            'neutral_score': 1,
            'confidence': 0
        })

print("\nRunning FinBERT on all management utterances...")
sentiment_cols = mgmt_df['utterance'].apply(get_sentiment)
mgmt_df = pd.concat([mgmt_df.reset_index(drop=True), sentiment_cols], axis=1)
mgmt_df.to_csv("data/management_sentiment.csv", index=False)

print("\n✅ Done!")
print("\nSentiment breakdown:")
print(mgmt_df['sentiment'].value_counts())
print("\nBy company:")
print(mgmt_df.groupby(['ticker', 'sentiment']).size().unstack(fill_value=0))

In [ ]:
import matplotlib.pyplot as plt

quarter_order = ['Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25']
companies = sorted(mgmt_df['ticker'].unique())

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, company in enumerate(companies):
    data = mgmt_df[mgmt_df['ticker'] == company].groupby('quarter')[
        ['positive_score', 'negative_score', 'neutral_score']
    ].mean()
    data = data.reindex([q for q in quarter_order if q in data.index])

    axes[i].plot(data.index, data['positive_score'],
                 marker='o', color='green', label='Positive', linewidth=2)
    axes[i].plot(data.index, data['negative_score'],
                 marker='o', color='red', label='Negative', linewidth=2)
    axes[i].plot(data.index, data['neutral_score'],
                 marker='o', color='gray', label='Neutral',
                 linewidth=2, linestyle='--')
    axes[i].set_title(f'{company}', fontsize=11, fontweight='bold')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(fontsize=7)

# Hide unused subplots
for j in range(len(companies), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Management Sentiment Trends FY25 — All Companies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("data/sentiment_all_companies.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

In [ ]:
import os
import torch
import pandas as pd
from transformers import pipeline


device = "mps" if torch.backends.mps.is_available() else "cpu"
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=device
)
print(f"✅ FinBERT loaded on {device}")

In [ ]:
df = pd.read_csv("data/structured_utterances.csv")
mgmt_df = df[df['role'] == 'Management'].copy().reset_index(drop=True)
print(f"Running FinBERT on {len(mgmt_df)} utterances...")

def get_sentiment(text):
    try:
        text = str(text)[:512]
        scores = finbert(text)[0]
        score_dict = {s['label'].lower(): s['score'] for s in scores}
        dominant = max(score_dict, key=score_dict.get)
        return pd.Series({
            'sentiment': dominant,
            'positive_score': score_dict.get('positive', 0),
            'negative_score': score_dict.get('negative', 0),
            'neutral_score': score_dict.get('neutral', 0),
            'confidence': score_dict[dominant]
        })
    except:
        return pd.Series({
            'sentiment': 'neutral',
            'positive_score': 0,
            'negative_score': 0,
            'neutral_score': 1,
            'confidence': 0
        })

sentiment_cols = mgmt_df['utterance'].apply(get_sentiment)
mgmt_df = pd.concat([mgmt_df.reset_index(drop=True), sentiment_cols], axis=1)
mgmt_df.to_csv("data/management_sentiment.csv", index=False)

print("✅ Done!")
print("\nSentiment by company:")
print(mgmt_df.groupby(['ticker','sentiment']).size().unstack(fill_value=0))

In [ ]:
import matplotlib.pyplot as plt

quarter_order = ['Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25']
companies = sorted(mgmt_df['ticker'].unique())

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, company in enumerate(companies):
    data = mgmt_df[mgmt_df['ticker']==company].groupby('quarter')[
        ['positive_score','negative_score','neutral_score']
    ].mean()
    data = data.reindex([q for q in quarter_order if q in data.index])

    axes[i].plot(data.index, data['positive_score'],
                 marker='o', color='green', label='Positive', linewidth=2)
    axes[i].plot(data.index, data['negative_score'],
                 marker='o', color='red', label='Negative', linewidth=2)
    axes[i].plot(data.index, data['neutral_score'],
                 marker='o', color='gray', label='Neutral',
                 linewidth=2, linestyle='--')
    axes[i].set_title(f'{company}', fontsize=11, fontweight='bold')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(fontsize=7)

plt.suptitle('Management Sentiment Trends FY25 — 8 Companies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("data/sentiment_all_companies.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")